# BuildX Bootcamp — Part B1: IPL Ball-by-Ball Data Cleaning
Dataset: IPL Dataset 2008–2025 (ball-by-ball delivery data, ~283,679 rows)

This notebook loads the raw dataset, explores it, handles missing values, cleans text columns, fixes data types, removes duplicates, engineers a new column, and exports `cleaned_dataset.csv`.

In [ ]:
# 1. Load the dataset
# Upload the raw CSV to Colab (left sidebar -> Files -> Upload), then update the filename below.
import pandas as pd
import numpy as np

df = pd.read_csv("ipl_ball_by_ball.csv", low_memory=False)  # <-- change to your actual filename
# low_memory=False avoids the "mixed types" warning by reading each column in one pass
# rather than in chunks, so pandas can correctly infer a single dtype per column.
print("Shape:", df.shape)

# Safety check: if the raw CSV has duplicate column names (common in messy
# Kaggle exports), pandas will silently create repeated columns, which breaks
# later steps like .str.title(). Detect and fix that here.
if df.columns.duplicated().any():
    dupes = df.columns[df.columns.duplicated()].tolist()
    print("Duplicate column names found and removed (keeping first occurrence):", dupes)
    df = df.loc[:, ~df.columns.duplicated()]

print("Shape after dedup:", df.shape)
df.head()

In [ ]:
# 2. Display the first 5 rows
df.head()

In [ ]:
# Quick look at column names, dtypes, and non-null counts
df.info()

In [ ]:
# 3. Check missing values for every column
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]

In [ ]:
# 4. Handle missing values appropriately
# This is ball-by-ball data, so most "missing" values are not errors — they are
# structurally expected (e.g. a ball with no wicket has no dismissal info).

# Drop the unnamed/index artifact column if present (e.g. "Column1")
if "Column1" in df.columns:
    df = df.drop(columns=["Column1"])

# Columns that are only populated when a specific event happens on that ball:
# fill with a clear placeholder instead of dropping rows, since the absence
# of a wicket/extra/review IS the correct, meaningful value for that ball.
event_only_text_cols = [
    "extra_type", "wicket_kind", "player_out", "fielders",
    "review_batter", "team_reviewed", "review_decision",
    "umpires_call", "new_batter", "next_batter"
]
for col in event_only_text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")  # "None" = event did not occur on this ball

# Numeric event-only columns (e.g. runs_not_boundary, power_surge_start, team_wicket)
# get filled with 0, since a missing value here means the event count is zero.
event_only_numeric_cols = [
    "runs_not_boundary", "power_surge_start", "team_wicket",
    "bowler_wicket", "striker_out"
]
for col in event_only_numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Match-level metadata that's occasionally missing (e.g. method for matches with
# no DLS application, superover_winner for matches with no super over):
# fill with "Not Applicable" since blank here has a specific real-world meaning.
match_meta_cols = ["method", "superover_winner", "win_outcome", "match_won_by"]
for col in match_meta_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Not Applicable")

# For any remaining numeric columns with missing values, fill with the median
# (robust to outliers, e.g. extreme scores) rather than the mean.
remaining_num_cols = df.select_dtypes(include=[np.number]).columns
for col in remaining_num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Any remaining text columns: fill with "Unknown" rather than dropping rows,
# since dropping would lose otherwise-valid ball-by-ball records.
remaining_text_cols = df.select_dtypes(include=["object"]).columns
for col in remaining_text_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna("Unknown")

print("Remaining missing values:", df.isnull().sum().sum())

In [ ]:
# 5. Remove duplicate records
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows. New shape: {df.shape}")

In [ ]:
# 6. Clean all text columns: strip whitespace and standardise casing
text_cols = df.select_dtypes(include=["object"]).columns

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Team, venue, city, player names -> Title Case for consistent display
title_case_cols = ["batting_team", "bowling_team", "batter", "bowler", "non_striker",
                    "venue", "city", "player_out", "fielders", "umpire",
                    "toss_winner", "match_won_by", "player_of_match"]
for col in title_case_cols:
    if col in df.columns and isinstance(df[col], pd.Series):
        df[col] = df[col].astype(str).str.title()

# Categorical/flag-like columns -> lower case for consistent filtering in SQL/Power BI
lower_case_cols = ["match_type", "extra_type", "wicket_kind", "toss_decision",
                    "result_type", "gender", "team_type", "stage"]
for col in lower_case_cols:
    if col in df.columns and isinstance(df[col], pd.Series):
        df[col] = df[col].astype(str).str.lower()

df.head()

In [ ]:
# 7. Fix incorrect data types
# Date column: Excel often stores dates as serial numbers when exported to CSV.
# Try standard parsing first; fall back to Excel serial date origin if needed.
df["date"] = pd.to_datetime(df["date"], errors="coerce")
if df["date"].isnull().mean() > 0.5:  # most failed to parse -> likely Excel serial dates
    df["date"] = pd.to_datetime(df["date"], origin="1899-12-30", unit="D", errors="coerce")

# Boolean-like columns
bool_cols = ["valid_ball"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)

# Integer columns (ball-by-ball counters, runs, season, year etc.)
int_cols = ["match_id", "innings", "over", "ball", "bat_pos", "runs_batter",
            "balls_faced", "runs_extras", "runs_total", "runs_bowler",
            "runs_not_boundary", "non_striker_pos", "year", "day", "month",
            "team_runs", "team_balls", "team_wicket", "batter_runs", "batter_balls"]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

df.dtypes

In [ ]:
# 8. Rename columns for clearer, more consistent naming
rename_map = {
    "runs_batter": "runs_by_batter",
    "runs_bowler": "runs_conceded_by_bowler",
    "match_won_by": "winning_team",
    "player_out": "batter_dismissed",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
df.columns.tolist()

In [ ]:
# 9. Create a new useful column: classify each delivery's outcome
# This makes it much easier to do business analysis (e.g. dot-ball %, boundary %)
# in both SQL and Power BI without recomputing it every time.

def classify_ball(row):
    if row.get("wicket_kind", "none") not in ["none", "None", "nan"]:
        return "Wicket"
    runs = row.get("runs_by_batter", row.get("runs_batter", 0))
    if runs == 6:
        return "Six"
    elif runs == 4:
        return "Four"
    elif runs == 0:
        return "Dot Ball"
    else:
        return "Runs"

runs_col = "runs_by_batter" if "runs_by_batter" in df.columns else "runs_batter"
df["ball_outcome"] = df.apply(classify_ball, axis=1)

df["ball_outcome"].value_counts()

In [ ]:
# 10. Save the cleaned dataset
df.to_csv("cleaned_dataset.csv", index=False)
print("Saved cleaned_dataset.csv with shape:", df.shape)